In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q04/q04_rewrite/checkpoints/pre_cell_2.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Optimized for cudf
date_start = "1993-08-01"
date_end = "1993-11-01"

# Filter lineitem on GPU
flineitem = lineitem[lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE]

# Filter orders on GPU using string literals to avoid pd.Timestamp
forders = orders[(orders.O_ORDERDATE >= date_start) & (orders.O_ORDERDATE < date_end)]

# Use GPU‐accelerated merge instead of isin to reduce CPU work
keys = flineitem[["L_ORDERKEY"]].drop_duplicates()
jn = forders.merge(keys, left_on="O_ORDERKEY", right_on="L_ORDERKEY", how="inner")

# Group and count on GPU
total = jn.groupby("O_ORDERPRIORITY", as_index=False)["O_ORDERKEY"].count()

CPU times: user 71.4 ms, sys: 45.3 ms, total: 117 ms
Wall time: 137 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q04/rewritten/o4_mini_high_small_q04/checkpoints/post_cell_2_try_4.pickle